# W2Q1 — Customer Lifetime Journey
**Question:** How long does it take users to move through each funnel stage? Does paid vs organic acquisition affect journey speed?

**Audience:** Product & Marketing Teams  
**Data source:** `ANALYTICS.MARTS.MART_SUBSCRIPTION_FUNNEL`  
**SQL:** `sql/W2Q1_customer_lifetime_journey.sql`

---

> ⚠️ **Data note (DN-002):** Paid subscriptions were not available before Jan 2020.
> Analysis is restricted to users who signed up from Jan 2020 onward.

**Stage timing definitions:**
| Metric | Definition |
|--------|-----------|
| `days_signup_to_trial` | Days from account creation to free trial start |
| `days_trial_to_3m` | Days from trial start to 3m subscription |
| `days_3m_to_12m` | Days from 3m subscription to 12m upgrade |

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from src.connection import query

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130})
PALETTE = sns.color_palette('muted', 8)

## Load Data

In [ ]:
with open('../sql/W2Q1_customer_lifetime_journey.sql') as f:
    sql = f.read()

df = query(sql)
print(f"Rows: {len(df):,}  |  Columns: {list(df.columns)}")
df.head(3)

## Distribution of Days Between Funnel Stages

In [ ]:
stage_cols = {
    'days_signup_to_trial': 'Signup → Trial',
    'days_trial_to_3m':     'Trial → 3m Sub',
    'days_3m_to_12m':       '3m → 12m Sub',
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (col, label), color in zip(axes, stage_cols.items(), PALETTE):
    data = df[col].dropna()
    data = data[data >= 0]  # exclude negative values if any
    ax.hist(data, bins=40, color=color, edgecolor='white', linewidth=0.5)
    ax.axvline(data.median(), color='red', linestyle='--', linewidth=1.5, label=f'Median: {data.median():.0f}d')
    ax.set_title(label, fontsize=12)
    ax.set_xlabel('Days')
    ax.set_ylabel('Number of Users')
    ax.legend(fontsize=9)
    sns.despine(ax=ax)

fig.suptitle('Distribution of Days Between Funnel Stages', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../output/W2Q1_stage_duration_distributions.png', bbox_inches='tight')
plt.show()

## Paid vs Organic — Median Journey Time

In [ ]:
summary_rows = []
for col, label in stage_cols.items():
    for acq in ['paid', 'organic']:
        data = df[(df['acquisition_type'] == acq) & df[col].notna()][col]
        data = data[data >= 0]
        summary_rows.append({
            'Stage': label,
            'Acquisition': acq.title(),
            'N': len(data),
            'Median (days)': round(data.median(), 1),
            'P25 (days)': round(data.quantile(0.25), 1),
            'P75 (days)': round(data.quantile(0.75), 1),
            'Mean (days)': round(data.mean(), 1),
        })

summary = pd.DataFrame(summary_rows)
display(summary)

In [ ]:
pivot = summary.pivot(index='Stage', columns='Acquisition', values='Median (days)')

fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind='bar', ax=ax, color=PALETTE[:2], edgecolor='white')
ax.set_title('Median Days Between Funnel Stages — Paid vs Organic', fontsize=13, pad=12)
ax.set_xlabel('')
ax.set_ylabel('Median Days')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Acquisition')
sns.despine()
plt.tight_layout()
plt.savefig('../output/W2Q1_paid_vs_organic_journey.png', bbox_inches='tight')
plt.show()

## Funnel Stage Reach Rates

In [ ]:
total = len(df)
reach = pd.DataFrame({
    'Stage': ['Has Trial', 'Has 3m Sub', 'Has 12m Sub'],
    'Users': [
        df['has_trial'].sum(),
        df['has_3m_subscription'].sum(),
        df['has_12m_subscription'].sum(),
    ]
})
reach['% of Total'] = (reach['Users'] / total * 100).round(1)
display(reach)

## Findings

- **Signup → Trial timing:** ...
- **Trial → 3m timing:** ...
- **3m → 12m timing:** ...
- **Paid vs organic difference:** ...
- **Overall funnel reach:** ...
- **Recommendation:** ...